In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# Fresh kernel — Cell 1
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

In [3]:
spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .getOrCreate()

print("Spark version:", spark.version)  # confirm JVM is alive

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


In [4]:
# Read dimension user data
df_read_dim_user = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_user")

In [5]:
# Read raw user data
df_read_user_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_user.json")

In [6]:
# Extract the 'User_pk_sk' column and 'User_id' column from dim user data 
df_dim_user_id_and_pk = df_read_dim_user.select(F.col("User_pk_sk"),F.col("User_id").alias("dim_user_id"))

In [7]:
# Extract the 'friends' column and 'user_id' column from user data 

df_friend_user = df_read_user_data.select(F.col("friends"),F.col("user_id"))

In [8]:
# This function is used to clean friend IDs from the 'friends' column

@F.udf(returnType=ArrayType(StringType())) # Define DataFrame schema
def clean_friend_user(text):

    if text is None:
       return ["has no friends"] 
     
    list_of_ids = text.split(",")
    list_of_ids_without_space = [i.strip() for i in list_of_ids if i is not None]         

    if len(list_of_ids_without_space) == 0:
        return  ["has no friends"] 
    return list_of_ids_without_space



df_clean_friend_ids_col = df_friend_user.withColumn("friend_id", clean_friend_user(F.col("friends"))).drop(F.col("friends"))

df_clean_friend_ids_col.cache()
df_clean_friend_ids_col.count()

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


1987897

In [9]:
# Transfer column values to rows
df_friends_exploded =  df_clean_friend_ids_col.withColumn("friend_id_explode",F.explode(F.col("friend_id"))).drop(F.col("friend_id"))

In [14]:
# Match friend IDs against user IDs to look up friend surrogate/primary keys
df_user_friend_joined = df_friends_exploded.join(df_dim_user_id_and_pk, on=df_friends_exploded["friend_id_explode"] == df_dim_user_id_and_pk["dim_user_id"], how="inner").drop(F.col("dim_user_id")) 

In [24]:
# Rename columns to avoid naming conflicts
df_alias = df_user_friend_joined.select(F.col("user_id").alias("join_user_id") ,F.col("friend_id_explode").alias("join_friend_id_explode"),F.col("User_pk_sk").alias("join_friend_User_pk_sk") )

In [25]:
# Match user IDs against user dimension to look up user surrogate/primary keys

df_user_friends_with_keys = df_alias.join(df_dim_user_id_and_pk, on=df_alias["join_user_id"] == df_dim_user_id_and_pk["dim_user_id"], how="inner") 

In [28]:
df_select_user_friend_pk = df_user_friends_with_keys.select(F.col("join_friend_User_pk_sk").alias("Friend_pk"),F.col("User_pk_sk").alias("User_pk") )

In [ ]:
df_select_user_friend_pk.show()

In [ ]:
df_user_friends_with_keys.show()

In [ ]:
# test
#df_user_friends_with_keys.filter(F.col("user_id").contains("HEvyblFw4I-UsMqgPGYY_Q")).show(truncate=False)

+-----------+----------------------+---------+----------------+---------------+----------------+----------+-----------------+-------------+------------+-----------+-----------+--------------------+---------------------+----------------------+-------------------------+----------------------+----------------------+----------------------+-----------------------+----------------------+-----------------------+------------------------+------------------------+
|User_pk_sk |User_id               |User_name|first_elite_year|last_elite_year|user_elite_count|join_date |User_review_count|review_useful|review_funny|review_cool|review_fans|review_average_stars|review_compliment_hot|review_compliment_more|review_compliment_profile|review_compliment_cute|review_compliment_list|review_compliment_note|review_compliment_plain|review_compliment_cool|review_compliment_funny|review_compliment_writer|review_compliment_photos|
+-----------+----------------------+---------+----------------+---------------+---

In [ ]:
# test
#df_read_dim_user.filter(F.col("user_id").contains("---1lKK3aKOuomHnwAkAow")).show(truncate=False)

+-----------+----------------------+---------+----------------+---------------+----------------+----------+-----------------+-------------+------------+-----------+-----------+--------------------+---------------------+----------------------+-------------------------+----------------------+----------------------+----------------------+-----------------------+----------------------+-----------------------+------------------------+------------------------+
|User_pk_sk |User_id               |User_name|first_elite_year|last_elite_year|user_elite_count|join_date |User_review_count|review_useful|review_funny|review_cool|review_fans|review_average_stars|review_compliment_hot|review_compliment_more|review_compliment_profile|review_compliment_cute|review_compliment_list|review_compliment_note|review_compliment_plain|review_compliment_cool|review_compliment_funny|review_compliment_writer|review_compliment_photos|
+-----------+----------------------+---------+----------------+---------------+---

In [ ]:
#df_user_friends_with_keys.count()

14126166

In [ ]:
#df_join.count()

14366592

In [ ]:
#104302820-106104025

-1801205

In [ ]:
#df = df_dim_user_id_and_pk.withColumn(
 # "map_id_name",F.create_map(F.lit("User_sk"),
 #  F.col("User_pk_sk"), F.lit("dim_User_id"),F.col("User_id")) )#.drop()

#df_map = df_dim_user_id_and_pk.select(df_dim_user_id_and_pk.User_id.alias("user_id_map"),F.create_map([ df_dim_user_id_and_pk.User_id,df_dim_user_id_and_pk.User_pk_sk]).alias("user_map"))

In [23]:
#df_map.show()

In [ ]:
#df_join_explode_dim_user = df_friends_exploded.join(df_map, on=df_friends_exploded["user_id"] == df_map["user_id_map"], how="inner") 

In [25]:
#df = df_join_explode_dim_user.withColumn("friend_pk_sk" ,F.map_values(F.col("user_map")) )

In [ ]:
df.show()

In [ ]:
#df_join_explode_dim_user.show(truncate=False)

In [ ]:
#df_join_explode_dim_user.count()

104302820

In [ ]:
#df_join_explode_dim_user = df_friends_exploded.join(df_read_dim_user, on=df_friends_exploded["user_id"] == df_read_dim_user["user_id"], how="inner") \
  #  .withColumn("new_id", F.coalesce(df_friends_exploded["user_id"], df_read_dim_user["user_id"])).select(F.col("User_pk_sk"),df_friends_exploded["user_id"], df_friends_exploded["friend_id_explode"])#.drop(F.col("user_id"),F.col("tip_date"))

In [ ]:
#df_join_explode_dim_user_1 = df_join_explode_dim_user.select(df_join_explode_dim_user["friend_id_explode"],df_join_explode_dim_user["User_pk_sk"].alias("user_sk"),df_join_explode_dim_user["User_pk_sk"].alias("user_id_before_join"))

In [ ]:
#result =  df_join_explode_dim_user_1.join(df_read_dim_user, on=df_join_explode_dim_user_1["friend_id_explode"] == df_read_dim_user["user_id"], how="inner")
         #.select(df_join_explode_dim_user_1["friend_id_explode"],df_join_explode_dim_user_1["user_sk"],df_join_explode_dim_user_1["user_id_before_join"],df_read_dim_user["User_id"],df_read_dim_user["User_pk_sk"].alias("user_sk_after_join"))

In [ ]:
#result.show(truncate=False)

In [ ]:
#result.count()

14126166

In [ ]:
#result.count()

104302820

In [ ]:
#df_friends_exploded.count()

106104025

In [ ]:
#from pyspark.sql import functions as F

#df_clean_friend_ids_col.select(F.size(F.col("friend_id")).alias("friend_count")) \
 #   .summary("min", "25%", "50%", "75%", "max") \
   # .show()

+-------+------------+
|summary|friend_count|
+-------+------------+
|    min|           1|
|    25%|           1|
|    50%|           2|
|    75%|          46|
|    max|       14995|
+-------+------------+



In [ ]:
#df_read_user_data.columns

['average_stars',
 'compliment_cool',
 'compliment_cute',
 'compliment_funny',
 'compliment_hot',
 'compliment_list',
 'compliment_more',
 'compliment_note',
 'compliment_photos',
 'compliment_plain',
 'compliment_profile',
 'compliment_writer',
 'cool',
 'elite',
 'fans',
 'friends',
 'funny',
 'name',
 'review_count',
 'useful',
 'user_id',
 'yelping_since']

In [ ]:
#li = [" a ","baasf " ,]

In [ ]:
#for i  in li:
 #   print(i.split())

['a']
['baasf']
